# RTStream V2 Cookbook

Use VideoDB to connect a live RTSP feed, understand it continuously, turn the understanding into a searchable index, and react to events in real time.

The required path is:

**connect to VideoDB → connect the stream → create an understanding → read records → create an index → search → stop resources**

Alerts, transcription, pause/resume, and recording export are clearly marked as optional. Run the required sections from top to bottom. Live resources consume compute while they are running, so always run **Stop and clean up** before leaving the notebook.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/rtstream/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. Prerequisites**

You need a [VideoDB API key](https://console.videodb.io/) and an RTSP source. A public sample stream is included below.

**Install the RTStream V2 SDK**

Run this once per notebook session. If the notebook asks you to restart the kernel after installation, restart it and continue with the next section.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv

**Connect to VideoDB**

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
if not api_key:
    raise ValueError("A VideoDB API key is required.")

conn = connect(api_key=api_key, base_url="https://api.videodb.io")
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection ID:", collection.id)

**Choose an RTSP source**

RTSP (Real-Time Streaming Protocol) is commonly used by live cameras and streaming servers. VideoDB pulls media from the URL, so a custom source must be reachable from the internet—not `localhost` or a private network address.

Use either public sample below, or replace `RTSP_URL` with your own feed:

- Crib: `rtsp://samples.rts.videodb.io:8554/crib`
- Cricket: `rtsp://samples.rts.videodb.io:8554/cricket`
- Custom: `rtsp://your-public-host:port/path`

**Connect the RTStream**

`connect_rtstream()` connects the source and returns an RTStream with an `rts-` ID.

- `STORE_RECORDING` retains the live media so it can be exported after the stream stops. Set it to `False` when you do not need the recording.
- `INCLUDE_AUDIO` requests the source's audio track. Enable it only when the source has audio and you plan to use transcription.

In [ ]:
import time
from datetime import datetime, timezone

RTSP_URL = "rtsp://samples.rts.videodb.io:8554/crib"
# RTSP_URL = "rtsp://samples.rts.videodb.io:8554/cricket"
# RTSP_URL = "rtsp://your-public-host:port/path"
STREAM_NAME = "rtstream-v2-cookbook"

STORE_RECORDING = True   # Retain media so the stopped stream can be exported.
INCLUDE_AUDIO = False    # Enable only when the source has audio.

media_types = ["video", "audio"] if INCLUDE_AUDIO else ["video"]
run_started_at = time.time()

rtstream = collection.connect_rtstream(
    url=RTSP_URL,
    name=f"{STREAM_NAME}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    media_types=media_types,
    store=STORE_RECORDING,
)

print("RTStream ID:", rtstream.id)
print("Status:", rtstream.status)
print("Source:", RTSP_URL)
print("Media types:", media_types)

### **2. Understand the live stream**

**Create a continuous understanding**

Understanding continuously turns live video into time-aligned descriptions. For each window, VideoDB samples frames, analyzes them with the VLM and your prompt, and stores the result so it can power records, indexing, search, and alerts.

The settings in this cell apply specifically to `rtstream.understand()`:

- `WINDOW` is the duration of each understanding segment. `"10s"` means the VLM analyzes one 10-second portion of the stream at a time. Shorter windows produce updates more frequently.
- `FRAME_COUNT` is the number of frames sampled evenly across each window and sent to the VLM. With a 10-second window and 5 frames, samples are spaced about 2.5 seconds apart. More frames provide more visual context but require more processing.
- `SCENE_PROMPT` tells the VLM what to describe in every window.

The VLM writes each response to the named output `scene`. The 10-second/5-frame defaults are a practical starting point.

`store=True` is important in this workflow: it makes understanding records durable and allows an index to consume the output.

In [ ]:
WINDOW = "10s"          # Analyze the stream in 10-second segments.
FRAME_COUNT = 5          # Sample 5 frames evenly across each segment.
SCENE_PROMPT = "Describe the scene clearly. Mention whether a baby or crib is visible and what is happening."

understanding = rtstream.understand(
    segmentation={"type": "time", "window": WINDOW},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"frame_count": FRAME_COUNT},
            "config": {"prompt": SCENE_PROMPT},
        }
    ],
    store=True,
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
print("Available outputs:", list(understanding.outputs))
print("Scene output descriptor:", understanding.outputs["scene"])

**Retrieve an existing understanding**

In [ ]:
same_understanding = rtstream.get_understanding(understanding.id)
all_understandings = rtstream.list_understanding()

print("Fetched ID:", same_understanding.id)
print("Understanding IDs on this stream:", [item.id for item in all_understandings])

**Read durable understanding records**

Records remain available after the RTStream stops. If `records` is empty, wait for another window and rerun the cell.

In [ ]:
understanding_records = understanding.get_records(
    start=run_started_at,
    end=time.time(),
    output="scene",
)
understanding_records

### **3. Index the understanding**

An index turns a named understanding output into records that can be searched.

**Create an index**

`use_for=["semantic"]` enables semantic search on the `scene` output.

In [ ]:
index = rtstream.index(
    source=understanding.outputs["scene"],
    name="rtstream-v2-scenes",
    use_for=["semantic"],
)
index

**Retrieve existing indexes**

In [ ]:
same_index = rtstream.get_index(index.id)
all_indexes = rtstream.list_indexes()

same_index, all_indexes

**Read index records**

If `records` is empty, wait for another understanding window and rerun the cell.

In [ ]:
index_records = index.get_records(
    start=run_started_at,
    end=time.time(),
)
index_records

### **4. Search the live index**

Search uses natural language to find matching time ranges in the index.

**Search for a scene**

In [ ]:
SEARCH_QUERY = "a baby or crib is visible"

search_result = rtstream.search(
    query=SEARCH_QUERY,
    index_id=index.id,
    result_threshold=5,
)
shots = search_result.get_shots()
shots

**Play a result**

Run this after the search returns at least one shot.

In [ ]:
first_shot = shots[0]
first_shot.play()

## 5. Optional — create a live alert

An event defines the condition; an alert applies that event to this index. A callback URL is required for alert delivery.

Set `RTSTREAM_ALERT_CALLBACK_URL` in the environment or replace the empty value below with your webhook URL. This can be an ngrok URL forwarding to an HTTP receiver running beside the notebook. If no URL is supplied, the cell safely skips alert creation.

In [ ]:
EVENT_PROMPT = "A baby crib is visible in the frame"
EVENT_LABEL = "crib-visible"
CALLBACK_URL = os.getenv("RTSTREAM_ALERT_CALLBACK_URL", "").strip()

alert_id = None
event_id = None

if not CALLBACK_URL:
    print("Skipped. Set RTSTREAM_ALERT_CALLBACK_URL and rerun this cell.")
else:
    event_id = conn.create_event(event_prompt=EVENT_PROMPT, label=EVENT_LABEL)
    alert_id = index.create_alert(
        event_id=event_id,
        callback_url=CALLBACK_URL,
    )
    print("Event ID:", event_id)
    print("Alert ID:", alert_id)
    print("Alerts on this index:", index.list_alerts())

### Enable or disable an alert

A new alert is enabled when it is created. Use these methods to control delivery without deleting the index. This demonstration is opt-in so the active alert is not interrupted accidentally.

In [ ]:
RUN_ALERT_TOGGLE_DEMO = False

if not alert_id:
    print("Skipped because no alert was created.")
elif not RUN_ALERT_TOGGLE_DEMO:
    print("Alert remains enabled. Set RUN_ALERT_TOGGLE_DEMO=True to test disable and enable.")
else:
    index.disable_alert(alert_id)
    print("Alert disabled")
    index.enable_alert(alert_id)
    print("Alert enabled again")
    print(index.list_alerts())

## 6. Optional — live transcription

Run this section only if the RTSP source has audio and you set `INCLUDE_AUDIO=True` **before** creating the RTStream.

`start_transcript()` begins transcription and `get_transcript()` reads durable finalized segments. The cleanup section stops transcription automatically when it was started here.

In [ ]:
transcription_started = globals().get("transcription_started", False)
transcript_records = []

if not INCLUDE_AUDIO:
    print("Skipped. Set INCLUDE_AUDIO=True, then reconnect the RTStream under Prerequisites.")
else:
    if not transcription_started:
        transcript_status = rtstream.start_transcript(engine="assemblyai")
        transcription_started = True
        print("Transcription started:", transcript_status)

    transcript_response = rtstream.get_transcript(
        page=1,
        page_size=1000,
        start=run_started_at,
        end=time.time(),
        engine="assemblyai",
    )
    transcript_records = transcript_response.get("transcription_records", [])

    if transcript_records:
        print(f"Found {len(transcript_records)} transcript record(s)")
        for record in transcript_records[:5]:
            print(record)
    else:
        print("No transcript yet. Wait for speech and rerun this cell.")

## 7. Optional — pause and resume processing

An understanding and its index have independent lifecycles. Stopping either job pauses new processing but keeps existing records. This demonstration is opt-in because pausing the live pipeline creates a gap in new results.

In [ ]:
RUN_PAUSE_RESUME_DEMO = False

if not RUN_PAUSE_RESUME_DEMO:
    print("Set RUN_PAUSE_RESUME_DEMO=True to run this lifecycle demonstration.")
else:
    index.stop()
    understanding.stop()
    print("Paused index and understanding")

    understanding.start()
    index.start()
    print("Resumed understanding and index")
    print("Understanding status:", rtstream.get_understanding(understanding.id).status)
    print("Index status:", rtstream.get_index(index.id).status)

## 8. Stop and clean up — always run this

Run this cell even if an earlier step failed. It attempts each cleanup action independently in this order:

1. disable the alert;
2. stop transcription;
3. stop the index and understanding;
4. stop the RTStream.

Stopping compute does not delete stored understanding or index records.

In [ ]:
cleanup_results = {}

if globals().get("alert_id"):
    try:
        index.disable_alert(alert_id)
        cleanup_results["alert"] = "disabled"
    except Exception as exc:
        cleanup_results["alert"] = f"disable failed: {exc}"

if globals().get("transcription_started"):
    try:
        rtstream.stop_transcript(engine="assemblyai")
        cleanup_results["transcription"] = "stopped"
    except Exception as exc:
        cleanup_results["transcription"] = f"stop failed: {exc}"

for label, resource in (
    ("index", globals().get("index")),
    ("understanding", globals().get("understanding")),
    ("rtstream", globals().get("rtstream")),
):
    if resource is None:
        continue
    try:
        resource.stop()
        cleanup_results[label] = "stopped"
    except Exception as exc:
        cleanup_results[label] = f"stop failed: {exc}"

print(cleanup_results)

## 9. Verify stored data after stopping

These reads show the durability boundary: stopping live compute does not erase stored resources and records. Each read is isolated so one unavailable optional surface does not hide the others.

In [ ]:
def summarize(resource):
    return {
        "id": getattr(resource, "id", None),
        "name": getattr(resource, "name", None),
        "status": getattr(resource, "status", None),
    }


def safe_read(label, fetch):
    try:
        value = fetch()
        print(f"{label}: OK")
        return value
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        print(f"{label}: {error}")
        return {"error": error}


post_stop_end = time.time()
post_stop_reads = {
    "stream": safe_read("stream", lambda: summarize(collection.get_rtstream(rtstream.id))),
    "understandings": safe_read(
        "understandings",
        lambda: [summarize(item) for item in rtstream.list_understanding()],
    ),
    "understanding": safe_read(
        "understanding",
        lambda: summarize(rtstream.get_understanding(understanding.id)),
    ),
    "understanding_records": safe_read(
        "understanding records",
        lambda: understanding.get_records(
            start=run_started_at,
            end=post_stop_end,
            output="scene",
            page_size=100,
        ),
    ),
    "indexes": safe_read("indexes", lambda: [summarize(item) for item in rtstream.list_indexes()]),
    "index": safe_read("index", lambda: summarize(rtstream.get_index(index.id))),
    "index_records": safe_read(
        "index records",
        lambda: index.get_records(start=run_started_at, end=post_stop_end, page_size=100),
    ),
    "alerts": safe_read("alerts", lambda: index.list_alerts()),
    "transcript": safe_read(
        "transcript",
        lambda: rtstream.get_transcript(
            start=run_started_at,
            end=post_stop_end,
            page_size=100,
        ),
    ),
}

print("Post-stop reads complete. Use post_stop_reads to inspect full payloads.")

## 10. Optional — export the retained recording

This works only when `STORE_RECORDING=True`, and only after the RTStream is stopped. Recording finalization is asynchronous, so the cell retries for up to about 30 seconds.

Export is idempotent: rerunning it returns the same VideoDB asset rather than creating a duplicate.

In [ ]:
if not STORE_RECORDING:
    print("Skipped because this RTStream was created with STORE_RECORDING=False.")
else:
    export_result = None
    last_export_error = None

    for attempt in range(1, 7):
        try:
            export_result = rtstream.export(name="RTStream V2 Cookbook Recording")
            break
        except Exception as exc:
            last_export_error = exc
            print(f"Export attempt {attempt}/6 is not ready: {exc}")
            time.sleep(5)

    if export_result is None:
        print("The recording did not finalize during the retry window:", last_export_error)
    else:
        print("Video ID:", export_result.video_id)
        print("Duration:", export_result.duration)
        print("Media URL:", export_result.stream_url)
        print("Player URL:", export_result.player_url)

## Quick reference

| Goal | Public SDK call |
|---|---|
| Connect a live source | `collection.connect_rtstream(...)` |
| Start understanding | `rtstream.understand(...)` |
| Retrieve understandings | `rtstream.get_understanding(id)` / `rtstream.list_understanding()` |
| Read understanding output | `understanding.get_records(start, end, output="scene")` |
| Create a V2 index | `rtstream.index(source=understanding.outputs["scene"])` |
| Retrieve V2 indexes | `rtstream.get_index(id)` / `rtstream.list_indexes()` |
| Read indexed records | `index.get_records(start, end)` |
| Search | `rtstream.search(query, index_id=index.id)` |
| Create and control an alert | `index.create_alert(...)`, `disable_alert(id)`, `enable_alert(id)` |
| Start or stop transcription | `rtstream.start_transcript(...)` / `rtstream.stop_transcript(...)` |
| Read transcript segments | `rtstream.get_transcript(...)` |
| Generate a playable search result | `shot.generate_stream()` |
| Export a retained recording | `rtstream.export()` after `rtstream.stop()` |

## What you built

You connected a live source, created continuous understanding, stored its output in a searchable index, and read durable records.

You also searched live scenes and saw how optional alerts, transcription, clip generation, recording export, and lifecycle controls fit into the workflow.